###  핵심 포인트:
- 일반 문서는 더 어렵다: Q&A 형태가 아니므로 LLM이 질문을 "만들어내야" 합니다
- 좋은 프롬프트가 중요: 단계별 지시사항으로 품질을 높일 수 있습니다
- 도구(Tools) 활용: Agent에 도구를 추가하면 더 정교한 처리가 가능합니다

### 📚 FAQ 문서 vs 일반 문서",

| 특성 | FAQ 문서 | 일반 문서 |
|------|----------|----------|
| 구조 | Q&A 형태로 정리됨 | 서술형 텍스트 |
| 추출 난이도 | 쉬움 (파싱만 하면 됨) | 어려움 (질문 생성 필요) |
| 프롬프트 | 단순 추출 지시 | 단계별 상세 지시 필요 |
| 도구 필요성 | 낮음 | 높음 (요약, 분석 등) |


In [1]:
# setting
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from pydantic import BaseModel, Field
from typing import List

class GroundTruth(BaseModel):
    question: str = Field(description="The question asked by the user.")
    answer: str = Field(description="The correct answer to the user's question.")
    source: str = Field(description="The source document or reference for the answer.")  # FAQ 노트북에는 없던 필드

class GroundTruthList(BaseModel):
    ground_truths: List[GroundTruth] = Field(description="A list of ground truth question-answer pairs.")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(
        model="gemini-3-flash-preview",
        temperature=0
    )

In [4]:
from langchain.agents import create_agent

# 도구 없이 Agent 생성 - 프롬프트만으로 Q&A 생성을 수행
agent = create_agent(
    model=model,
    tools=[],  # 도구 없이 사용 (순수 LLM 추론만)
    response_format=GroundTruthList,  # Pydantic 모델로 구조화된 출력 강제
    system_prompt="""You are an expert at generating ground truth datasets for evaluating AI agents. 
You will be given a document from which you will generate the ground truth data.
Follow these steps below to perform the task 

1. Read the provided document carefully.
2. Think about the purpose of the document and the type of information it contains.
3. Generate a list of question-aswer pairs that accurately reflect the content of the document.
4. If you want to mention the document, make sure to include the title""",
)

In [6]:
from pathlib import Path

# 직원 복리후생 가이드 문서 (FAQ가 아닌 서술형 문서)
# 가이드 문서: employee_benefits_and_welfare_guide.md (서술형)
markdown_path = Path("documents_with_english_titles_md/employee_benefits_and_welfare_guide.md")
markdown_content = markdown_path.read_text(encoding="utf-8")

In [7]:
# Agent에 문서를 전달하여 Q&A 생성 실행
benefits_dataset = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"""Generate a ground truth dataset based on the following document:
{markdown_content}
"""
    }]
})

In [8]:
# tool 을 활용하지 않은 버전
import pandas as pd

# structured_response에서 Q&A 리스트 추출
df = pd.DataFrame([gt.model_dump() for gt in benefits_dataset['structured_response'].ground_truths])

# CSV로 저장
df.to_csv("./welfare_list_with_agent.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} Q&A pairs to output/welfare_list_with_agent.csv")
df

Saved 10 Q&A pairs to output/welfare_list_with_agent.csv


,question,answer,source
0,직원 복리후생 및 복지 가이드에 따르면 복리후생 제도의 적용 대상은 누구인가요?,"정규직, 계약직, 인턴 등 모든 직원에게 적용됩니다.",직원 복리후생 및 복지 가이드
1,회사가 제공하는 정기 건강 검진의 비용과 항목은 어떻게 되나요?,"검진 비용은 회사가 100% 부담하며, 항목은 혈액 검사와 X-ray를 포함한 기본...",직원 복리후생 및 복지 가이드
2,근속 연수가 5년인 직원이 받을 수 있는 연차 유급 휴가 일수는 며칠인가요?,근속 연수 4~6년에 해당하므로 17일의 연차 휴가가 지급됩니다.,직원 복리후생 및 복지 가이드
3,육아 휴가의 기간과 급여 지원 조건은 무엇인가요?,"육아 휴가는 최대 1년까지 가능하며, 첫 3개월 동안은 급여의 50%를 지원합니다.",직원 복리후생 및 복지 가이드
4,본인 결혼 시 제공되는 경조 휴가와 경조사 지원금은 각각 얼마인가요?,본인 결혼 시 5일의 경조 휴가와 30만 원의 지원금이 제공됩니다.,직원 복리후생 및 복지 가이드
5,외부 교육 프로그램 수강 시 회사가 지원하는 금액의 한도는 얼마인가요?,"등록 비용의 50%를 지원하며, 연간 최대 한도는 200,000원입니다.",직원 복리후생 및 복지 가이드
6,문화의 날 혜택에 대해 설명해 주세요.,"매월 마지막 금요일 오후 3시 이후에 자유 시간이 부여되며, 영화나 공연 관람 등을...",직원 복리후생 및 복지 가이드
7,매월 지원되는 통신비와 식사비의 최대 금액은 각각 얼마인가요?,"통신비는 매월 50,000원, 식사비는 월 최대 150,000원까지 지원됩니다.",직원 복리후생 및 복지 가이드
8,복리후생 신청은 어떤 경로를 통해 해야 하나요?,사내 HR 시스템을 통해 신청할 수 있습니다.,직원 복리후생 및 복지 가이드
9,복리후생 관련 문의를 위한 인사팀 담당자의 연락처는 무엇인가요?,"담당자는 김지현 인사팀장이며, 이메일은 kimjh@company.com, 전화번호는...",직원 복리후생 및 복지 가이드


## Custom agent with tools
* analyze_document: 문서를 요약하여 핵심 내용을 파악
* generate_qna_list: 문서 내용과 요약을 기반으로 Q&A 쌍 생성
* Agent가 먼저 문서를 분석(요약)한 후, 그 요약을 참고하여 Q&A를 생성하는 2단계 프로세스를 수행 -> 더 정교한 Q&A 생성 가능

In [9]:
# === 커스텀 도구 정의 ===
# Agent가 사용할 도구들을 @tool 데코레이터로 정의합니다

from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

@tool
def generate_qna_list(markdown_content: str, summary: str) -> GroundTruthList:
    """
    문서 내용과 요약을 받아 Q&A 리스트를 생성합니다.
    
    Args:
        markdown_content: 원본 마크다운 문서 내용
        summary: analyze_document 도구로 생성된 요약
    
    Returns:
        GroundTruthList: 질문-답변 쌍의 리스트
    """
    # 도구 내부에서 사용할 프롬프트
    # 요약 정보를 활용하여 더 나은 Q&A를 생성합니다
    system_prompt = """You are a helpful assistant that generates a list of questions and answers from the given markdown file. 
Follow the steps below in order to generate the list of questions and answers:
1. Look at the summary the content of the markdown file to understand the purpose of the document.
2. Generate a list of questions and answers based on the summarized content.
3. Make sure to format the question and answer in Korean as the original question and answer is in Korean."""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Text: {markdown_content}\nSummary: {summary}"),  # 요약 정보 포함
    ])
    model_with_json = model.with_structured_output(GroundTruthList)
    qna_chain = prompt | model_with_json
    return qna_chain.invoke({"markdown_content": markdown_content, "summary": summary})


@tool
def analyze_document(text: str) -> str:
    """
    문서를 분석하고 요약을 반환합니다.
    
    이 도구는 generate_qna_list 전에 호출되어
    문서의 핵심 내용을 파악하는 데 사용됩니다.
    
    Args:
        text: 분석할 문서 텍스트
    
    Returns:
        str: 문서 요약
    """
    return model.invoke(f"""{text}

summarize the document above""")

In [10]:
# 도구를 포함한 Agent 생성
# 시스템 프롬프트에서 두 도구의 사용 방법을 명시적으로 안내합니다
agent_with_tools = create_agent(
    model=model,
    tools=[analyze_document, generate_qna_list],  # 문서 분석 + Q&A 생성 도구
    response_format=GroundTruthList,
    system_prompt="""You are an expert at generating ground truth datasets for evaluating AI agents. 
You will be given a document from which you will generate the ground truth data.
You have access to two tools: analyze_document and generate_qna_list.
The analyze_document tool helps you summarize the document, while the generate_qna_list tool generates the Q&A pairs based on the document and its summary.
Use these tools effectively to create a comprehensive ground truth dataset.""",
)

In [11]:
# Agent가 analyze_document → generate_qna_list 순서로 도구를 호출하여 Q&A를 생성합니다
benefit_with_tools_dataset = agent_with_tools.invoke({
    "messages": [{
        "role": "user",
        "content": f"""Generate a ground truth dataset based on the following document:
{markdown_content}
"""
    }]
})

In [12]:
# === Agent 결과를 CSV로 저장 ===
import pandas as pd

# structured_response에서 Q&A 리스트 추출
df = pd.DataFrame([gt.model_dump() for gt in benefit_with_tools_dataset['structured_response'].ground_truths])

# CSV로 저장
df.to_csv("./welfare_list_with_agent_tools.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} Q&A pairs to output/welfare_list_with_agent_tools.csv")
df

Saved 10 Q&A pairs to output/welfare_list_with_agent_tools.csv


,question,answer,source
0,이 복리후생 가이드의 적용 대상은 누구인가요?,"정규직, 계약직, 인턴을 포함한 모든 직원에게 적용됩니다.",직원 복리후생 및 복지 가이드 제 2 조
1,회사가 제공하는 건강 관리 혜택에는 무엇이 있나요?,"국민건강보험료 일부 부담, 단체 실손 보험 제공(입원 및 통원 치료비 지원), 그리...","직원 복리후생 및 복지 가이드 제 3 조, 제 4 조"
2,근속 연수가 5년인 직원은 1년에 며칠의 연차 휴가를 받나요?,근속 연수 4~6년 구간에 해당하므로 17일의 연차 유급 휴가를 받습니다.,직원 복리후생 및 복지 가이드 제 5 조
3,육아 휴직 시 급여 지원 조건은 어떻게 되나요?,"육아 휴직은 최대 1년까지 가능하며, 첫 3개월 동안 급여의 50%를 지원합니다.",직원 복리후생 및 복지 가이드 제 6 조
4,본인 생일이나 결혼 기념일에도 휴가를 쓸 수 있나요?,"네, 경조 휴가 규정에 따라 본인 생일과 결혼 기념일에 각각 1일의 휴가가 부여됩니다.",직원 복리후생 및 복지 가이드 제 7 조
5,"유연 근무제(재택 근무, 시차 출근)를 사용하려면 어떤 절차가 필요한가요?",사전 승인을 받은 직원이 본인 승인을 통해 적용하여 사용할 수 있습니다.,직원 복리후생 및 복지 가이드 제 8 조
6,자기계발을 위한 교육비 지원 한도는 얼마인가요?,"외부 교육 프로그램 등록 시 비용의 50%(연간 최대 200,000원)를 지원하며,...",직원 복리후생 및 복지 가이드 제 9 조
7,'문화의 날' 혜택에 대해 설명해 주세요.,"매월 마지막 금요일 오후 3시 이후에 자유 시간이 부여되며, 영화나 공연 관람 등을...",직원 복리후생 및 복지 가이드 제 12 조
8,매달 지급되는 통신비와 식사 지원금은 각각 얼마인가요?,"통신비는 매월 50,000원, 식사비는 월 최대 150,000원까지 지원됩니다.","직원 복리후생 및 복지 가이드 제 15 조, 제 17 조"
9,복리후생 신청 방법과 문의처는 어디인가요?,"사내 HR 시스템을 통해 신청할 수 있으며, 문의 사항은 인사팀 김지현 팀장(02-...","직원 복리후생 및 복지 가이드 제 18 조, 제 19 조"


- 위에서는 하나의 문서(employee_benefits_and_welfare_guide.md)로 테스트
- 이제 documents_with_english_titles_markdown/ 폴더의 모든 마크다운 파일에 대해 Q&A를 생성하고, 하나의 CSV로 통합
- 각 Q&A의 source 필드에는 원본 파일명이 기록되어, 나중에 평가 시 어떤 문서에서 나온 질문인지 추적 가능



In [14]:
# === 모든 마크다운 파일에 대해 Golden Dataset 생성 (agent_with_tools 사용) ===
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# 모든 마크다운 파일 경로 수집
markdown_dir = Path("documents_with_english_titles_md")
markdown_files = sorted(markdown_dir.glob("*.md"))

print(f"Found {len(markdown_files)} markdown files:")
for f in markdown_files:
    print(f"  - {f.name}")

# 누적 결과를 저장할 리스트
all_ground_truths = []

# 각 마크다운 파일에 대해 처리
for md_file in tqdm(markdown_files, desc="Processing markdown files"):
    try:
        # 파일 내용 읽기
        markdown_content = md_file.read_text(encoding="utf-8")
        
        # Agent를 사용하여 Golden Dataset 생성
        dataset = agent.invoke({
            "messages": [{
                "role": "user",
                "content": f"""Generate a ground truth dataset based on the following document:
{markdown_content}
"""
            }]
        })
        
        # structured_response에서 Q&A 리스트 추출
        ground_truths = dataset['structured_response'].ground_truths
        
        # 각 Q&A의 source 정보 업데이트 (파일명으로 설정)
        for gt in ground_truths:
            gt.source = md_file.stem  # 파일명(확장자 제외)을 source로 설정
            all_ground_truths.append(gt.model_dump())
        
        print(f"✓ {md_file.name}: {len(ground_truths)} Q&A pairs generated")
        
    except Exception as e:
        print(f"✗ Error processing {md_file.name}: {str(e)}")

# 모든 데이터를 하나의 DataFrame으로 통합
cumulative_df = pd.DataFrame(all_ground_truths)

# CSV로 저장
output_file = "cumulative_golden_dataset.csv"
cumulative_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\n✓ Saved {len(cumulative_df)} total Q&A pairs to {output_file}")
print(f"\nDataset summary by source:")
print(cumulative_df['source'].value_counts())
print(f"\nFirst few rows:")
cumulative_df

Found 7 markdown files:
  - delegation_of_authority.md
  - employee_benefits_and_welfare_faq.md
  - employee_benefits_and_welfare_guide.md
  - employee_handbook_and_hr_policy.md
  - expense_management_guide.md
  - it_support_guide.md
  - legal_and_compliance_policy.md


Processing markdown files:  14%|█▍        | 1/7 [00:09<00:54,  9.06s/it]

✓ delegation_of_authority.md: 10 Q&A pairs generated


Processing markdown files:  29%|██▊       | 2/7 [00:16<00:40,  8.11s/it]

✓ employee_benefits_and_welfare_faq.md: 13 Q&A pairs generated


...

Processing markdown files:  86%|████████▌ | 6/7 [01:00<00:10, 10.92s/it]

✓ it_support_guide.md: 10 Q&A pairs generated


Processing markdown files: 100%|██████████| 7/7 [01:07<00:00,  9.70s/it]

✓ legal_and_compliance_policy.md: 10 Q&A pairs generated

✓ Saved 73 total Q&A pairs to cumulative_golden_dataset.csv

Dataset summary by source:
source
employee_benefits_and_welfare_faq      13
delegation_of_authority                10
employee_benefits_and_welfare_guide    10
employee_handbook_and_hr_policy        10
expense_management_guide               10
it_support_guide                       10
legal_and_compliance_policy            10
Name: count, dtype: int64

First few rows:


,question,answer,source
0,전결 규정 (Delegation of Authority)의 목적은 무엇인가요?,회사 내 의사 결정의 신속성과 책임성을 확보하기 위하여 전결 권한을 명확히 정하는 ...,delegation_of_authority
1,"인사(채용, 승진, 징계) 사항에 대한 최종 승인권자는 누구입니까?",인사 사항에 대한 최종 승인권자는 대표이사입니다.,delegation_of_authority
2,50만원 이하의 예산 승인은 누가 할 수 있나요?,"팀장과 부서장이 승인할 수 있으며, 본부장에게는 보고가 이루어집니다.",delegation_of_authority
3,200만원을 초과하는 예산 승인의 결재 라인은 어떻게 되나요?,"팀장 검토, 부서장 검토, 본부장 승인을 거쳐 대표이사가 최종 승인합니다.",delegation_of_authority
4,500만원을 초과하는 계약 체결 시 본부장의 역할은 무엇인가요?,"본부장은 해당 계약 건에 대해 승인 역할을 수행하며, 이후 대표이사의 최종 승인이 ...",delegation_of_authority
...,...,...,...
68,직장 내 성희롱 예방 교육은 얼마나 자주 실시해야 하나요?,회사는 매년 성희롱 예방 교육을 실시해야 합니다.,legal_and_compliance_policy
69,법률 및 준수 정책에 명시된 징계의 종류는 무엇인가요?,"징계의 종류에는 경고, 감봉, 정직, 해고가 있습니다.",legal_and_compliance_policy
70,징계 절차에서 대상자의 권리는 무엇인가요?,징계 대상자는 위반 행위에 대해 소명할 수 있는 기회를 가집니다.,legal_and_compliance_policy
71,직원이 법적 의무나 권리에 대해 상담하고 싶을 때 어느 부서에 문의해야 하나요?,인사팀 또는 법무팀에 상담할 수 있습니다.,legal_and_compliance_policy
